# Vision Naturalistic Baseline



In [1]:
# This setup cell has no substantive output by itself.
# It imports the naturalistic baseline runner and the helpers used to summarize the saved CSVs.

from pathlib import Path
import json
import pandas as pd

from vision.run_naturalistic_baseline import get_model_specs, run_naturalistic_baseline

In [2]:
# This cell defines the benchmark configuration.
# INCLUDE_CLIP=True means the naturalistic baseline compares the original ViT, ConvNeXt,
# and CLIP image tower in the same pipeline.

DEVICE = "cpu"
BATCH_SIZE = 16
SEED = 0
INCLUDE_CLIP = True
MODEL_SPECS = get_model_specs(include_clip=INCLUDE_CLIP)
OUTPUT_DIR = Path("results/vision/naturalistic")

## Fruits

In [3]:
# This cell runs the fruits baseline.
# The summary output reports the saved artifact paths plus top-level benchmark information.
# Task definition in this notebook:
# - identification: noisy stored image -> exact stored target
# - generalization: held-out real image -> stored image with highest human similarity

fruits_summary = run_naturalistic_baseline(
    category="fruits",
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    dataset_pkl="datasets_peterson.pkl",
    image_root=".",
    model_specs=MODEL_SPECS,
)
fruits_summary

{'category': 'fruits',
 'chance_accuracy_values': [1.25],
 'required_exemplars': None,
 'split_mode': 'balanced_exemplar_folds',
 'storage_sizes': [80],
 'n_seeds': None,
 'models': ['vit_base_patch16_224::cls',
  'vit_base_patch16_224::mean_tokens',
  'convnext_tiny::auto',
  'vit_base_patch16_clip_224.openai::cls',
  'vit_base_patch16_clip_224.openai::mean_tokens'],
 'sanity_all_pass': True,
 'best_generalization': {'vit_base_patch16_224::cls': {'category': 'fruits',
   'split_label': 'fold_2',
   'split_seed': 2,
   'split_mode': 'balanced_exemplar_folds',
   'model_name': 'vit_base_patch16_224',
   'pooling': 'cls',
   'layer': 'layer_11',
   'n_stored': 80,
   'n_probe': 40,
   'chance_accuracy': 1.25,
   'ident_accuracy': 100.0,
   'ident_avg_target_sim': 0.9974550186848786,
   'ident_avg_margin': 0.05835537162167862,
   'gen_accuracy': 47.5,
   'gen_avg_target_sim': 0.932647384322044,
   'gen_avg_margin': -0.008146042945666726,
   'gen_avg_retrieved_human_similarity': 0.72425,
 

In [4]:
# This table summarizes the fruits benchmark by model, pooling, and layer.
# Rows: one aggregated representation choice (model + pooling + layer).
# Fields:
# - ident_accuracy: mean exact-image retrieval accuracy
# - gen_accuracy: mean human-similarity generalization accuracy
# - gen_avg_margin: average gap between the correct stored target and the best wrong one
# - human_rdm_spearman: alignment between model geometry and human similarity geometry
# - gen_same_concept_accuracy: how often the retrieved stored image is from the same concept family
# Main finding: late layers retrieve best, while mid layers align best to human similarity.

fruits = pd.read_csv(OUTPUT_DIR / "fruits_combined.csv")
fruits.groupby(["model_name", "pooling", "layer"]).agg({
    "ident_accuracy": "mean",
    "gen_accuracy": "mean",
    "gen_avg_margin": "mean",
    "human_rdm_spearman": "mean",
    "gen_same_concept_accuracy": "mean",
}).sort_values("gen_accuracy", ascending=False).head(12)

ident_accuracy  \
model_name                       pooling     layer                      
vit_base_patch16_224             mean_tokens layer_11      100.000000   
vit_base_patch16_clip_224.openai cls         layer_11      100.000000   
convnext_tiny                    auto        layer_3       100.000000   
vit_base_patch16_224             cls         layer_11      100.000000   
vit_base_patch16_clip_224.openai mean_tokens layer_11      100.000000   
convnext_tiny                    auto        layer_2       100.000000   
                                             layer_0        99.166667   
vit_base_patch16_224             mean_tokens layer_6       100.000000   
vit_base_patch16_clip_224.openai mean_tokens layer_6       100.000000   
vit_base_patch16_224             cls         layer_6       100.000000   
                                 mean_tokens layer_0       100.000000   
vit_base_patch16_clip_224.openai cls         layer_6        49.166667   

                                                       gen_accuracy  \
model_name                       pooling     layer                    
vit_base_patch16_224             mean_tokens layer_11     48.333333   
vit_base_patch16_clip_224.openai cls         layer_11     46.666667   
convnext_tiny                    auto        layer_3      45.833333   
vit_base_patch16_224             cls         layer_11     44.166667   
vit_base_patch16_clip_224.openai mean_tokens layer_11     37.500000   
convnext_tiny                    auto        layer_2      22.500000   
                                             layer_0      14.166667   
vit_base_patch16_224             mean_tokens layer_6      14.166667   
vit_base_patch16_clip_224.openai mean_tokens layer_6      11.666667   
vit_base_patch16_224             cls         layer_6      10.833333   
                                 mean_tokens layer_0       7.500000   
vit_base_patch16_clip_224.openai cls         layer_6       6.666667   

                                                       gen_avg_margin  \
model_name                       pooling     layer                      
vit_base_patch16_224             mean_tokens layer_11       -0.015178   
vit_base_patch16_clip_224.openai cls         layer_11       -0.007951   
convnext_tiny                    auto        layer_3        -0.031299   
vit_base_patch16_224             cls         layer_11       -0.010002   
vit_base_patch16_clip_224.openai mean_tokens layer_11       -0.015778   
convnext_tiny                    auto        layer_2        -0.000222   
                                             layer_0        -0.004893   
vit_base_patch16_224             mean_tokens layer_6        -0.069697   
vit_base_patch16_clip_224.openai mean_tokens layer_6        -0.069563   
vit_base_patch16_224             cls         layer_6        -0.017068   
                                 mean_tokens layer_0        -0.031433   
vit_base_patch16_clip_224.openai cls         layer_6        -0.027223   

                                                       human_rdm_spearman  \
model_name                       pooling     layer                          
vit_base_patch16_224             mean_tokens layer_11            0.338123   
vit_base_patch16_clip_224.openai cls         layer_11            0.579635   
convnext_tiny                    auto        layer_3             0.297747   
vit_base_patch16_224             cls         layer_11            0.263785   
vit_base_patch16_clip_224.openai mean_tokens layer_11            0.481256   
convnext_tiny                    auto        layer_2             0.377717   
                                             layer_0             0.189642   
vit_base_patch16_224             mean_tokens layer_6             0.379343   
vit_base_patch16_clip_224.openai mean_tokens layer_6             0.387395   
vit_base_patch16_224             cls         layer_6             0.297806   
                                 mean_tokens layer_0             0.249127   
vit_base_patc

## Vegetables

In [5]:
# This cell runs the vegetables baseline on the balanced 3-exemplar subset.
# The summary output confirms that the same naturalistic pipeline was applied to vegetables
# after dropping the irregular concepts that break the balanced fold structure.

vegetables_summary = run_naturalistic_baseline(
    category="vegetables",
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    dataset_pkl="datasets_peterson.pkl",
    image_root=".",
    required_exemplars=3,
    model_specs=MODEL_SPECS,
)
vegetables_summary

{'category': 'vegetables',
 'chance_accuracy_values': [1.2820512820512822],
 'required_exemplars': 3,
 'split_mode': 'balanced_exemplar_folds',
 'storage_sizes': [78],
 'n_seeds': None,
 'models': ['vit_base_patch16_224::cls',
  'vit_base_patch16_224::mean_tokens',
  'convnext_tiny::auto',
  'vit_base_patch16_clip_224.openai::cls',
  'vit_base_patch16_clip_224.openai::mean_tokens'],
 'sanity_all_pass': True,
 'best_generalization': {'vit_base_patch16_224::cls': {'category': 'vegetables',
   'split_label': 'fold_2',
   'split_seed': 2,
   'split_mode': 'balanced_exemplar_folds',
   'model_name': 'vit_base_patch16_224',
   'pooling': 'cls',
   'layer': 'layer_11',
   'n_stored': 78,
   'n_probe': 39,
   'chance_accuracy': 1.2820512820512822,
   'ident_accuracy': 100.0,
   'ident_avg_target_sim': 0.9971911439579468,
   'ident_avg_margin': 0.05684995040204006,
   'gen_accuracy': 53.84615384615385,
   'gen_avg_target_sim': 0.93566022171443,
   'gen_avg_margin': -0.003939512681418282,
   'ge

In [6]:
# This table summarizes vegetables in the same format as fruits.
# Interpret the fields exactly the same way as the fruits table.
# Main finding: the fruits pattern replicates here too.
# In particular, late layers lead retrieval and mid layers lead human-alignment scores.

vegetables = pd.read_csv(OUTPUT_DIR / "vegetables_combined.csv")
vegetables.groupby(["model_name", "pooling", "layer"]).agg({
    "ident_accuracy": "mean",
    "gen_accuracy": "mean",
    "gen_avg_margin": "mean",
    "human_rdm_spearman": "mean",
    "gen_same_concept_accuracy": "mean",
}).sort_values("gen_accuracy", ascending=False).head(12)

ident_accuracy  \
model_name                       pooling     layer                      
vit_base_patch16_224             cls         layer_11           100.0   
                                 mean_tokens layer_11           100.0   
convnext_tiny                    auto        layer_3            100.0   
vit_base_patch16_clip_224.openai cls         layer_11           100.0   
                                 mean_tokens layer_11           100.0   
convnext_tiny                    auto        layer_2            100.0   
vit_base_patch16_clip_224.openai mean_tokens layer_6            100.0   
vit_base_patch16_224             mean_tokens layer_6            100.0   
                                 cls         layer_6            100.0   
convnext_tiny                    auto        layer_0            100.0   
vit_base_patch16_224             mean_tokens layer_0            100.0   
vit_base_patch16_clip_224.openai mean_tokens layer_0            100.0   

                                                       gen_accuracy  \
model_name                       pooling     layer                    
vit_base_patch16_224             cls         layer_11     49.572650   
                                 mean_tokens layer_11     46.153846   
convnext_tiny                    auto        layer_3      45.299145   
vit_base_patch16_clip_224.openai cls         layer_11     45.299145   
                                 mean_tokens layer_11     39.316239   
convnext_tiny                    auto        layer_2      32.478632   
vit_base_patch16_clip_224.openai mean_tokens layer_6      26.495726   
vit_base_patch16_224             mean_tokens layer_6      23.931624   
                                 cls         layer_6      23.076923   
convnext_tiny                    auto        layer_0       9.401709   
vit_base_patch16_224             mean_tokens layer_0       9.401709   
vit_base_patch16_clip_224.openai mean_tokens layer_0       8.547009   

                                                       gen_avg_margin  \
model_name                       pooling     layer                      
vit_base_patch16_224             cls         layer_11       -0.007381   
                                 mean_tokens layer_11       -0.016580   
convnext_tiny                    auto        layer_3        -0.033458   
vit_base_patch16_clip_224.openai cls         layer_11       -0.017313   
                                 mean_tokens layer_11       -0.010555   
convnext_tiny                    auto        layer_2        -0.000135   
vit_base_patch16_clip_224.openai mean_tokens layer_6        -0.034765   
vit_base_patch16_224             mean_tokens layer_6        -0.037967   
                                 cls         layer_6        -0.012857   
convnext_tiny                    auto        layer_0        -0.004217   
vit_base_patch16_224             mean_tokens layer_0        -0.029090   
vit_base_patch16_clip_224.openai mean_tokens layer_0        -0.060223   

                                                       human_rdm_spearman  \
model_name                       pooling     layer                          
vit_base_patch16_224             cls         layer_11            0.303827   
                                 mean_tokens layer_11            0.315389   
convnext_tiny                    auto        layer_3             0.321496   
vit_base_patch16_clip_224.openai cls         layer_11            0.493626   
                                 mean_tokens layer_11            0.492285   
convnext_tiny                    auto        layer_2             0.368353   
vit_base_patch16_clip_224.openai mean_tokens layer_6             0.460097   
vit_base_patch16_224             mean_tokens layer_6             0.461542   
                                 cls         layer_6             0.354688   
convnext_tiny                    auto        layer_0             0.284686   
vit_base_patch16_224             mean_tokens layer_0             0.327619   
vit_base_patc

## Animals

In [7]:
# This cell runs the animals baseline using random storage curves instead of exemplar folds.
# Animals does not have the same visible 3-exemplar concept structure as fruits/vegetables,
# so this category is evaluated across multiple random stored-set sizes and seeds.

animals_summary = run_naturalistic_baseline(
    category="animals",
    device=DEVICE,
    batch_size=BATCH_SIZE,
    seed=SEED,
    output_dir=OUTPUT_DIR,
    dataset_pkl="datasets_peterson.pkl",
    image_root=".",
    split_mode="random_storage_curve",
    storage_sizes=[40, 80],
    n_seeds=5,
    model_specs=MODEL_SPECS,
)
animals_summary

{'category': 'animals',
 'chance_accuracy_values': [1.25, 2.5],
 'required_exemplars': None,
 'split_mode': 'random_storage_curve',
 'storage_sizes': [40, 80],
 'n_seeds': 5,
 'models': ['vit_base_patch16_224::cls',
  'vit_base_patch16_224::mean_tokens',
  'convnext_tiny::auto',
  'vit_base_patch16_clip_224.openai::cls',
  'vit_base_patch16_clip_224.openai::mean_tokens'],
 'sanity_all_pass': True,
 'best_generalization': {'vit_base_patch16_224::cls': {'category': 'animals',
   'split_label': 'stored_80_seed_2',
   'split_seed': 2,
   'split_mode': 'random_storage_curve',
   'model_name': 'vit_base_patch16_224',
   'pooling': 'cls',
   'layer': 'layer_11',
   'n_stored': 80,
   'n_probe': 40,
   'chance_accuracy': 1.25,
   'ident_accuracy': 100.0,
   'ident_avg_target_sim': 0.9958626759660654,
   'ident_avg_margin': 0.04188455122482477,
   'gen_accuracy': 70.0,
   'gen_avg_target_sim': 0.9464424723121064,
   'gen_avg_margin': 0.013994442624434167,
   'gen_avg_retrieved_human_similarity'

In [8]:
# This table summarizes the animals benchmark by model, pooling, layer, and n_stored.
# Rows: one representation choice plus one stored-set size.
# Extra animal-specific fields:
# - n_stored: how many images were put in memory for that random split
# - gen_avg_retrieved_human_similarity: average human similarity of the retrieved item
# - gen_avg_human_similarity_regret: how much human similarity was lost relative to the best target
# Main finding: animals gives the strongest retrieval regime in the project.
# It also shows that late-layer retrieval can become positively separated from competitors.

animals = pd.read_csv(OUTPUT_DIR / "animals_combined.csv")
animals.groupby(["model_name", "pooling", "layer", "n_stored"]).agg({
    "ident_accuracy": "mean",
    "gen_accuracy": "mean",
    "gen_avg_margin": "mean",
    "human_rdm_spearman": "mean",
    "gen_avg_retrieved_human_similarity": "mean",
    "gen_avg_human_similarity_regret": "mean",
}).sort_values("gen_accuracy", ascending=False).head(12)

ident_accuracy  \
model_name                       pooling     layer    n_stored                   
vit_base_patch16_224             cls         layer_11 80                 100.0   
                                 mean_tokens layer_11 80                 100.0   
convnext_tiny                    auto        layer_3  80                 100.0   
vit_base_patch16_clip_224.openai cls         layer_11 40                 100.0   
                                                      80                 100.0   
vit_base_patch16_224             cls         layer_11 40                 100.0   
convnext_tiny                    auto        layer_3  40                 100.0   
vit_base_patch16_224             mean_tokens layer_11 40                 100.0   
vit_base_patch16_clip_224.openai mean_tokens layer_11 40                 100.0   
                                                      80                 100.0   
convnext_tiny                    auto        layer_2  40                 100.0   
                                                      80                 100.0   

                                                                gen_accuracy  \
model_name                       pooling     layer    n_stored                 
vit_base_patch16_224             cls         layer_11 80               63.00   
                                 mean_tokens layer_11 80               57.50   
convnext_tiny                    auto        layer_3  80               57.50   
vit_base_patch16_clip_224.openai cls         layer_11 40               56.75   
                                                      80               53.00   
vit_base_patch16_224             cls         layer_11 40               52.75   
convnext_tiny                    auto        layer_3  40               51.00   
vit_base_patch16_224             mean_tokens layer_11 40               50.25   
vit_base_patch16_clip_224.openai mean_tokens layer_11 40               46.75   
                                                      80               42.00   
convnext_tiny                    auto        layer_2  40               32.00   
                                                      80               27.50   

                                                                gen_avg_margin  \
model_name                       pooling     layer    n_stored                   
vit_base_patch16_224             cls         layer_11 80              0.010480   
                                 mean_tokens layer_11 80              0.017868   
convnext_tiny                    auto        layer_3  80              0.044627   
vit_base_patch16_clip_224.openai cls         layer_11 40              0.012405   
                                                      80              0.007916   
vit_base_patch16_224             cls         layer_11 40              0.006301   
convnext_tiny                    auto        layer_3  40              0.028005   
vit_base_patch16_224             mean_tokens layer_11 40              0.012614   
vit_base_patch16_clip_224.openai mean_tokens layer_11 40             -0.004531   
                                                      80             -0.009058   
convnext_tiny                    auto        layer_2  40             -0.000088   
                                                      80             -0.000091   

                                                                human_rdm_spearman  \
model_name                       pooling     layer    n_stored                       
vit_base_patch16_224             cls         layer_11 80                  0.105264   
                                 mean_tokens layer_11 80                  0.187760   
convnext_tiny                    auto        layer_3  80                  0.046470   
vit_base_patch16_clip_224.openai cls         layer_11 40                  0.549651   
                                                      80                  0.549651   
vit_base_patch16_224             cls       

## Integrity And Cross-Category Summary

In [9]:
# This JSON output is a dataset integrity check.
# It confirms that the Peterson filenames match the local image folders and records category sizes,
# image dimensions, and any filtering decisions used to build the naturalistic tasks.

integrity = json.loads((OUTPUT_DIR / "dataset_integrity.json").read_text())
integrity

{'animals': {'category': 'animals',
  'n_images': 120,
  'matrix_shape': [120, 120],
  'is_symmetric': True,
  'diag_mean': 1.0,
  'missing_on_disk': [],
  'extra_on_disk': [],
  'n_concepts': 1,
  'concept_histogram': {'stim-': 120},
  'image_size': [300, 300],
  'required_exemplars': None,
  'dropped_concepts': []},
 'fruits': {'category': 'fruits',
  'n_images': 120,
  'matrix_shape': [120, 120],
  'is_symmetric': True,
  'diag_mean': 1.0,
  'missing_on_disk': [],
  'extra_on_disk': [],
  'n_concepts': 40,
  'concept_histogram': {'ackee': 3,
   'apple': 3,
   'apricot': 3,
   'atemoya': 3,
   'babaco': 3,
   'bacuri': 3,
   'banana': 3,
   'blackberry': 3,
   'blueberry': 3,
   'cantaloupe': 3,
   'carambola': 3,
   'cherry': 3,
   'coconut': 3,
   'cranberry': 3,
   'date': 3,
   'deadmansfinger': 3,
   'dragonfruit': 3,
   'durian': 3,
   'fig': 3,
   'grape': 3,
   'guava': 3,
   'kiwi': 3,
   'lemon': 3,
   'lime': 3,
   'lychee': 3,
   'mango': 3,
   'orange': 3,
   'papaya': 3

In [10]:
# This cross-category table is sorted by generalization accuracy.
# Rows: the top-performing representation choices within fruits, vegetables, and animals.
# Key fields:
# - category: dataset category being summarized
# - model_name / pooling / layer: representation choice
# - gen_accuracy: top retrieval performance on the human-similarity task
# - human_rdm_spearman: human-alignment score for that same row
# Main finding: animals is the strongest retrieval category, while the late-layer retrieval pattern
# replicates across all three naturalistic categories.

comparison = pd.read_csv(OUTPUT_DIR / "animals_vs_fruits_vs_vegetables_summary.csv")
comparison.sort_values(["category", "gen_accuracy"], ascending=[True, False]).head(18)

,category,model_name,pooling,layer,ident_accuracy,gen_accuracy,gen_avg_margin,gen_avg_retrieved_human_similarity,gen_avg_human_similarity_regret,gen_same_concept_accuracy,human_rdm_spearman,chance_accuracy
31,animals,vit_base_patch16_224,cls,layer_11,100.000,57.875000,0.008390,0.801563,0.074094,NaN,0.105265,1.875
40,animals,vit_base_patch16_clip_224.openai,cls,layer_11,100.000,54.875000,0.010161,0.825444,0.050213,NaN,0.549651,1.875
38,animals,convnext_tiny,auto,layer_3,100.000,54.250000,0.036316,0.779050,0.096606,NaN,0.046470,1.875
34,animals,vit_base_patch16_224,mean_tokens,layer_11,100.000,53.875000,0.015241,0.790175,0.085481,NaN,0.187760,1.875
43,animals,vit_base_patch16_clip_224.openai,mean_tokens,layer_11,100.000,44.375000,-0.006795,0.735831,0.139825,NaN,0.360985,1.875
37,animals,convnext_tiny,auto,layer_2,100.000,29.750000,-0.000090,0.636288,0.239369,NaN,0.300918,1.875
35,animals,vit_base_patch16_224,mean_tokens,layer_6,100.000,20.875000,-0.047784,0.538806,0.336850,NaN,0.423428,1.875
44,animals,vit_base_patch16_clip_224.openai,mean_tokens,layer_6,100.000,20.750000,-0.051145,0.514306,0.361350,NaN,0.376055,1.875
32,animals,vit_base_patch16_224,cls,layer_6,99.500,12.500000,-0.017135,0.489553,0.386103,NaN,0.287819,1.875
42,animals,vit_base_patch16_clip_224.openai,mean_tokens,layer_0,100.000,7.375000,-0.080863,0.344681,0.530975,NaN,0.168027,1.875


- *Adding noise on the decision process*: After calculating the distances, add noise to the distance measures. Possibly trying out DAM.
- Ways to distrupt the identification process: Black out big chunks of the image. Partial input pattern. Try different approachs to image warping.

In [11]:
# This cross-category table is sorted by human_rdm_spearman instead of generalization accuracy.
# It answers a different question: which rows are most human-like geometrically, even if they
# are not the very best retrieval rows.
# Main finding: the best alignment rows are usually different from the best retrieval rows,
# which is the core late-layer retrieval vs mid-layer alignment split in this project.

comparison.sort_values(["category", "human_rdm_spearman"], ascending=[True, False]).head(18)

,category,model_name,pooling,layer,ident_accuracy,gen_accuracy,gen_avg_margin,gen_avg_retrieved_human_similarity,gen_avg_human_similarity_regret,gen_same_concept_accuracy,human_rdm_spearman,chance_accuracy
40,animals,vit_base_patch16_clip_224.openai,cls,layer_11,100.000,54.875000,0.010161,0.825444,0.050213,NaN,0.549651,1.875
35,animals,vit_base_patch16_224,mean_tokens,layer_6,100.000,20.875000,-0.047784,0.538806,0.336850,NaN,0.423428,1.875
44,animals,vit_base_patch16_clip_224.openai,mean_tokens,layer_6,100.000,20.750000,-0.051145,0.514306,0.361350,NaN,0.376055,1.875
43,animals,vit_base_patch16_clip_224.openai,mean_tokens,layer_11,100.000,44.375000,-0.006795,0.735831,0.139825,NaN,0.360985,1.875
37,animals,convnext_tiny,auto,layer_2,100.000,29.750000,-0.000090,0.636288,0.239369,NaN,0.300918,1.875
32,animals,vit_base_patch16_224,cls,layer_6,99.500,12.500000,-0.017135,0.489553,0.386103,NaN,0.287819,1.875
33,animals,vit_base_patch16_224,mean_tokens,layer_0,100.000,5.750000,-0.033336,0.329800,0.545856,NaN,0.197562,1.875
34,animals,vit_base_patch16_224,mean_tokens,layer_11,100.000,53.875000,0.015241,0.790175,0.085481,NaN,0.187760,1.875
36,animals,convnext_tiny,auto,layer_0,79.125,6.250000,-0.004605,0.339066,0.536591,NaN,0.175832,1.875
42,animals,vit_base_patch16_clip_224.openai,mean_tokens,layer_0,100.000,7.375000,-0.080863,0.344681,0.530975,NaN,0.168027,1.875


## Corruption Sweep

These cells summarize the follow-up experiment that makes the identification task harder by corrupting the probe images more aggressively.


In [ ]:
CORRUPTION_OUTPUT_DIR = Path("results/vision/naturalistic_corruption")
corruption_summary_path = CORRUPTION_OUTPUT_DIR / "corruption_sweep_summary.json"
corruption_rows_path = CORRUPTION_OUTPUT_DIR / "corruption_sweep_rows.csv"
corruption_summary = json.loads(corruption_summary_path.read_text()) if corruption_summary_path.exists() else {}
corruption_rows = pd.read_csv(corruption_rows_path) if corruption_rows_path.exists() else pd.DataFrame()
corruption_summary


In [ ]:
# This table highlights the most useful non-ceiling identification regimes.
# Rows are grouped corruption settings that keep identification below saturation while still
# leaving the generalization task measurable.
if corruption_rows.empty:
    pd.DataFrame()
else:
    (corruption_rows
        .groupby(["category", "corruption_label", "model_name", "pooling", "layer"], as_index=False)
        .agg({
            "ident_accuracy": "mean",
            "gen_accuracy": "mean",
            "gen_avg_margin": "mean",
            "human_rdm_spearman": "mean",
        })
        .query("40 <= ident_accuracy <= 90")
        .sort_values(["category", "gen_accuracy"], ascending=[True, False])
        .head(30))


In [ ]:
# This cell lists the preview images saved for the corruption sweep.
# Each preview PNG shows clean vs corrupted probes side by side for one corruption regime.
sorted(str(path) for path in CORRUPTION_OUTPUT_DIR.glob("**/previews/*.png"))[:20]


## Corruption Spot Checks

These cells summarize the new identification-hardening spot checks that were added after the meeting concern about ceiling identification accuracy.


In [ ]:
SPOTCHECK_DIR = Path("results/vision/naturalistic_spotcheck")
spotcheck_summary_path = SPOTCHECK_DIR / "spotcheck_summary.csv"
spotcheck = pd.read_csv(spotcheck_summary_path) if spotcheck_summary_path.exists() else pd.DataFrame()
spotcheck


In [ ]:
# This table is the fastest way to explain the new result.
# Each row is one manually chosen corruption regime. The key columns are:
# - best_gen_ident_accuracy: identification accuracy on the strongest retrieval layer
# - best_gen_accuracy: generalization accuracy on that same retrieval-focused row
# - best_align_ident_accuracy: identification accuracy on the strongest alignment layer
# Main finding so far: strong occlusion plus optional decision noise can finally break the 100% identification ceiling.
spotcheck[[
    "category",
    "corruption_label",
    "decision_noise_std",
    "best_gen_layer",
    "best_gen_ident_accuracy",
    "best_gen_accuracy",
    "best_align_layer",
    "best_align_ident_accuracy",
    "best_align_accuracy",
]].sort_values(["category", "best_gen_ident_accuracy"]) if not spotcheck.empty else pd.DataFrame()


In [ ]:
sorted(str(path) for path in SPOTCHECK_DIR.glob("**/previews/*.png"))
